# 6.4 Je eigen vectoren — wat is één rij, wanneer een rij "haha" is?

De bloemencache van 6.3 werkte omdat één rij één bloem was: een duidelijke foto, een zuiver label, één keer geëncodeerd. Chatdata biedt die luxe niet. Een groot deel van de berichten telt vijf woorden of minder, en een bericht van drie woorden embedt naar iets wat vrijwel inhoudsloos is — niet omdat de encoder slecht is, maar omdat er simpelweg nauwelijks iets is om te encoderen.

Dit notebook zet dat falen in scène, lost het op, en controleert of de oplossing daadwerkelijk werkte — eerst op de IRC-showcase, waar zes productieve auteurs de controles iets geven om tegen af te zetten, en daarna op je eigen chat, met dezelfde functies.

> Vereist `torch`, `sentence-transformers` en `transformers`: eenmalig lokaal `uv sync --extra huggingface`.

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from notebooktester import param

from goad_toolkit.visualizer import HeatmapPlot, PlotSettings, ProjectionPlot
from scripts.pipelines import build_irc_pipeline
from scripts.sessionize import fit_session_threshold, merge_messages, sessionize
from wa_analyzer.data import load_own_chat, load_showcase

EMBEDDER = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


## 6.4.1 Zet de klont (blob) in scène

Zes van de meest actieve posters van #ubuntu-uk, één rij per bericht, geëmbed precies zoals ze getypt werden. Als een individueel bericht veel signaal zou dragen over wie het schreef, zou een 2D-projectie nu al zes clusters moeten tonen — dezelfde claim die 6.3 controleerde en bevestigde voor bloemenfoto's.

In [ ]:
irc = build_irc_pipeline().apply(load_showcase("ubuntu_irc"))
uk = irc[(irc.channel == "#ubuntu-uk") & (~irc.is_action)].copy()
uk["timestamp"] = (
    pd.to_datetime(uk["date"]) + pd.to_timedelta(uk["hh"], unit="h") + pd.to_timedelta(uk["mm"], unit="m")
)
uk = uk.sort_values(["author", "timestamp"]).reset_index(drop=True)

TOP_AUTHORS = uk.author.value_counts().head(6).index.tolist()
print(f"{len(uk):,} messages, median {uk.message.str.split().str.len().median():.0f} words, "
      f"top authors: {TOP_AUTHORS}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

rng = np.random.default_rng(42)
BLOB_SAMPLE = param(1500, test=150)
sample = uk[uk.author.isin(TOP_AUTHORS)].sample(n=BLOB_SAMPLE, random_state=42)

X = EMBEDDER.encode(sample.message.tolist(), batch_size=64, show_progress_bar=False)
coords = PCA(n_components=2).fit_transform(X)

settings = PlotSettings(
    figsize=(7, 6),
    title="1,500 individual messages, six authors — embedded one at a time",
    xlabel="",
    ylabel="",
)
fig, ax = ProjectionPlot(settings).plot(coordinates=coords, labels=sample.author.to_numpy(),
                                        palette="tab10", s=10)

print(f"silhouette by author: {silhouette_score(X, sample.author):.3f}  (0 = no structure)")


Een vormloze klont, en een silhouette binnen afrondingsfout van nul — precies zoals het bloemenplaatje van 6.3 eruitzag *vóór* PCA, behalve dat dit *ná* embedding is. De encoder faalde niet; "thanks!" en "lol" dragen in drie woorden simpelweg niet de identiteit van zes mensen met zich mee. **Dit is een 'wat-is-één-rij'-probleem — fase 2 van de `goad_analysis_checklist`-methode, dezelfde vraag waarmee les 1 opende — niet een probleem met het model.**

## 6.4.2 Drie manieren om één rij te definiëren

Voeg opeenvolgende berichten van dezelfde auteur samen tot één eenheid, op drie verschillende manieren:

1. **Vast venster (Fixed window)** — elke 15 berichten van één auteur, waar ze toevallig ook over gaan.
2. **Per-auteur-per-dag** — alles wat één persoon op één kalenderdag zei.
3. **Tijdsinterval-sessies (Time-gap sessions)** — een nieuwe eenheid zodra het tijdsverschil sinds het vorige bericht van die persoon groter is dan een drempelwaarde. Die drempelwaarde moet niet gegokt worden.

Les 4 fitte een exponentiële verdeling op de pauzes tussen berichten, beperkt tot intervallen korter dan een uur — het regime binnen een burst — en las die fit als bewijs voor *wanneer* mensen in bursts praten in plaats van in een regelmatig tempo. Dezelfde fit, op deze data, geeft het getal dat sessievorming daadwerkelijk nodig heeft: de intervalduur waarboven een vervolg meer lijkt op toeval dan op een direct antwoord.

In [ ]:
threshold = fit_session_threshold(uk["timestamp"], uk["author"])
print(f"session threshold: {threshold:.0f}s ({threshold / 60:.1f} minutes)")

uk["session_id"] = sessionize(uk, "timestamp", "author", threshold)
subset = uk[uk.author.isin(TOP_AUTHORS)].copy()

K = 15
subset["window_id"] = subset.groupby("author").cumcount() // K
subset["day"] = subset["timestamp"].dt.date

fixed_units = merge_messages(subset, ["author", "window_id"])
daily_units = merge_messages(subset, ["author", "day"])
session_units = merge_messages(subset, ["author", "session_id"])

for name, units in [("fixed-window", fixed_units), ("per-day", daily_units), ("session", session_units)]:
    words = units.message.str.split().str.len()
    print(f"{name:>12s}: {len(units):>6,} units, median {words.median():>4.0f} words each")


~15 minuten, afgelezen van de fit in plaats van gegokt. Drie verdedigbare manieren om te antwoorden op "wat is één rij" — en de volgende sectie controleert of het antwoord er werkelijk toe doet, in plaats van alleen maar drie verschillende getallen te bewonderen.

## 6.4.3 De samenvoeging controleren, niet alleen bewonderen

Drie verdedigbare eenheden, en geen ground truth om ze tegen te scoren: handmatig geannoteerde antwoordlinks bestaan voor IRC (`jkkummerfeld/irc-disentanglement`), maar voor `#ubuntu`, niet voor de twee kanalen in de showcase van deze cursus. De controle is dus self-supervised, en heeft helemaal geen annotatie nodig: splits elke eenheid in tweeën op tijd, embed de twee helften afzonderlijk, en vraag of een nearest-neighbour-zoektocht op cosinus-similariteit de echte andere helft vaker vindt dan toeval. Een samenvoegstrategie die inhoudelijk coherente eenheden oplevert moet hier met gemak voor slagen; eentje die ruis produceert zou het niet beter moeten doen dan toeval.

In [ ]:
def self_retrieval_accuracy(frame: pd.DataFrame, group_cols: list[str], min_messages: int = 6,
                            n_eval: int = 400, seed: int = 42,
                            single_message: bool = False) -> tuple[float, int]:
    """Split each unit's messages in half by time, embed both halves, and check whether
    cosine similarity finds the true other half more often than chance.

    With `single_message=True` each half is one message rather than several joined together
    -- one from early in the unit against one from later -- which is the raw-message baseline.
    """
    sizes = frame.groupby(group_cols).size()
    keys = sizes[sizes >= min_messages].index.tolist()
    rng = np.random.default_rng(seed)
    if len(keys) > n_eval:
        keys = [keys[i] for i in rng.choice(len(keys), size=n_eval, replace=False)]

    first, second = [], []
    grouped = frame.sort_values("timestamp").groupby(group_cols)
    for key in keys:
        grp = grouped.get_group(key)
        half = len(grp) // 2
        if single_message:
            first.append(grp.message.iloc[0])
            second.append(grp.message.iloc[half])
        else:
            first.append(" ".join(grp.message.iloc[:half]))
            second.append(" ".join(grp.message.iloc[half:]))

    A = EMBEDDER.encode(first, batch_size=64, show_progress_bar=False)
    B = EMBEDDER.encode(second, batch_size=64, show_progress_bar=False)
    A = A / np.linalg.norm(A, axis=1, keepdims=True)
    B = B / np.linalg.norm(B, axis=1, keepdims=True)
    nearest = (A @ B.T).argmax(axis=1)
    accuracy = float((nearest == np.arange(len(keys))).mean())
    return accuracy, len(keys)


Losse ruwe berichten kunnen niet doormidden gesplitst worden, dus hun baseline is dezelfde controle met `single_message=True`: één bericht van vroeg in een burst tegenover eentje van later in dezelfde burst, in plaats van een tekst die doormidden gesneden is.

In [ ]:
N_EVAL = param(400, test=40)

results = {"single message": self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL,
                                                     single_message=True)}
results["per-day"] = self_retrieval_accuracy(subset, ["author", "day"], n_eval=N_EVAL)
results["fixed-window"] = self_retrieval_accuracy(subset, ["author", "window_id"], n_eval=N_EVAL)
results["session"] = self_retrieval_accuracy(subset, ["author", "session_id"], n_eval=N_EVAL)

print(f"{'strategy':>14s}  {'accuracy':>9s}  {'n pairs':>8s}   vs. chance")
for name, (acc, n) in results.items():
    chance = 1 / n if n else float("nan")
    print(f"{name:>14s}  {acc:>8.1%}   {n:>7,}   {acc / chance:>5.0f}x")


Elke samenvoegstrategie verslaat een enkel bericht met ruime marge, en **tijdsinterval-sessievorming wint overtuigend** — de eenheden die het produceert worden het meest betrouwbaar teruggevonden bij hun andere helft puur op basis van betekenis. Vaste vensters hakken gedachtes doormidden bij bericht 15 ongeacht wat er gebeurt; per-dag-eenheden verdunnen één echt gesprek tussen alle andere afzonderlijke gesprekken die die persoon die dag had. De grens goed krijgen, op de manier zoals §6.4.2 het fitte, is geen detail — het is het verschil tussen een bruikbare eenheid en een verdunde eenheid.

## 6.4.4 Cosinus-similariteit op samengevoegde eenheden

Twee dingen waar een samengevoegd, geëmbed corpus goed voor is: vinden wat lijkt op één specifiek ding, en vragen of "lijkt op" overeenkomt met iets wat je al kunt controleren.

In [ ]:
units = session_units[session_units.n_messages >= 5].reset_index(drop=True)
vectors = EMBEDDER.encode(units.message.tolist(), batch_size=64, show_progress_bar=False)
normed = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

rng = np.random.default_rng(7)
query = rng.integers(0, len(units))
similarity = normed @ normed[query]
ranked = np.argsort(similarity)
nearest, farthest = ranked[-2], ranked[0]  # -1 is the query itself

for label, i in [("QUERY", query), ("NEAREST", nearest), ("FARTHEST", farthest)]:
    print(f"{label} ({units.author.iloc[i]}, cosine={similarity[i]:.2f}):")
    print(f"  {units.message.iloc[i][:200]}")
    print()


De dichtstbijzijnde sessie gaat over hetzelfde soort onderwerp als de zoekopdracht — beide gaan over meer halen uit beperkt geheugen, één voor swap en één voor een GPU — en de verste is een totaal ander gesprek, over het routeren van audio over een netwerk. Cosinus-similariteit op samengevoegde eenheden zoekt op onderwerp, wat precies is wat ervan gevraagd werd en wat individuele berichten niet konden ondersteunen.

In [ ]:
centroids = pd.DataFrame(
    {author: normed[units.author.to_numpy() == author].mean(axis=0) for author in TOP_AUTHORS}
).T
centroids = centroids.div(np.linalg.norm(centroids, axis=1), axis=0)
author_similarity = pd.DataFrame(
    centroids.to_numpy() @ centroids.to_numpy().T, index=TOP_AUTHORS, columns=TOP_AUTHORS
)

settings = PlotSettings(figsize=(6, 5), title="Author centroids, cosine similarity", xlabel="", ylabel="")
fig, ax = HeatmapPlot(settings).plot(data=author_similarity, fmt=".2f")


Elk paar ligt op 0,82–0,94 — hoog, en nauwelijks onderscheidend tussen zes verschillende mensen. Dat is geen bug in de methode; het is wat een **semantische** embedding hoort te doen. Iedereen in dit kanaal bespreekt Ubuntu-ondersteuning, dus het centroïde van iedereen wijst grofweg dezelfde kant op. De dialect-vingerafdrukken uit les 5 — `:-)` tegenover `:)`, ☺ voor één persoon en niemand anders, weggelaten apostroffen — scheidden de acht drukste mensen in ditzelfde kanaal kraakhelder, maar dat signaal is *stilistisch*, gecodeerd in interpunctie en woordkeuze, niet in *betekenis*. Een embeddingmodel meet het ding waarvoor het getraind is, en dat is inhoud, geen gewoonte. Grijp naar het juiste gereedschap: trigram-tellingen voor stijl, embeddings voor onderwerp.

## 6.4.5 Laat de baseline racen tegen de embedding

6.2 bouwde een baseline die niets kost om te fitten: karakter-trigram-tellingen. Verslaat een nergens op gefinetunede sentence-embedding deze baseline op exact dezelfde samengevoegde eenheden bij de ene taak waarin hij een voordeel zou moeten hebben — zes mensen uit elkaar houden op basis van wat ze schreven?

In [ ]:
PER_AUTHOR_CAP = param(600, test=80)
balanced = units.groupby("author", group_keys=False).apply(
    lambda g: g.sample(min(len(g), PER_AUTHOR_CAP), random_state=42), include_groups=False
).join(units[["author"]])
train, test = train_test_split(balanced, test_size=0.3, stratify=balanced.author, random_state=42)

trigrams = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
Xtr_cv = trigrams.fit_transform(train.message)
Xte_cv = trigrams.transform(test.message)
acc_trigram = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_cv, train.author).predict(Xte_cv))

Xtr_emb = EMBEDDER.encode(train.message.tolist(), batch_size=64, show_progress_bar=False)
Xte_emb = EMBEDDER.encode(test.message.tolist(), batch_size=64, show_progress_bar=False)
acc_embed = accuracy_score(test.author, LogisticRegression(max_iter=1000).fit(Xtr_emb, train.author).predict(Xte_emb))

baseline = test.author.value_counts(normalize=True).max()
print(f"majority-class baseline : {baseline:.1%}")
print(f"trigram counts + logreg : {acc_trigram:.1%}")
print(f"sentence embedding + logreg : {acc_embed:.1%}")


De goedkope, handmatig ontworpen, niet-voorgetrainde trigramvector **wint met ruime voorsprong** — beide verslaan met gemak het gokken op de meerderheidsauteur, maar het tellen van karakters identificeert deze zes mensen veel beter dan een algemene sentence-embedding doet. Dit is de bevinding uit §6.4.4 opnieuw vanuit een andere hoek: de embedding is goed in *wát er gezegd werd*, en identiteit zit hier in *hóé*. Op zes gelabelde klassen en een paar duizend rijen is het interpreteerbare model geen noodoplossing voor wanneer je het chique model niet kunt betalen — voor deze taak is het het betere instrument, en dat weet je alleen door ze tegen elkaar te laten racen.

## 6.4.6 Een sentimentscore, als afsluiter

Nog één transfer-learned model, kort: een sentiment-classifier, gedraaid op dezelfde samengevoegde sessie-eenheden. Geen projectie, geen PCA — alleen een label en een betrouwbaarheid per eenheid, geaggregeerd per auteur.

In [ ]:
from transformers import pipeline as hf_pipeline

sentiment_model = hf_pipeline(  # ty: ignore[no-matching-overload]
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    top_k=None, truncation=True, max_length=512,
)

SENTIMENT_SAMPLE = param(60, test=10)
sample_units = units.sample(n=SENTIMENT_SAMPLE, random_state=0)
scores = []
for _, row in sample_units.iterrows():
    result = {item["label"]: item["score"] for item in sentiment_model(row.message[:512])[0]}
    scores.append({"author": row.author, **result})

sentiment = pd.DataFrame(scores)
sentiment.groupby("author")[["positive", "neutral", "negative"]].mean().round(2)


Een tabel, geen claim — zestig eenheden is bij lange na niet genoeg om iets te zeggen over iemands karakter of humeur, en de "negatieve" score van een helpdesk-kanaal betreft vooral *de problemen van andere mensen*, niet het humeur van de spreker. Behandel dit zoals 5.4 een losse p-waarde behandelde: een getal, nog geen bevinding, totdat er een mechanisme en replicatie achter zit.

## 6.4.7 Jouw beurt

Dezelfde functies, je eigen chat. `fit_session_threshold` en `sessionize` weten niet en geven er niet om dat de data ooit IRC was — ze verwachten een timestamp-kolom en een author-kolom, en die heeft jouw eigen export ook.

In [ ]:
own = load_own_chat()

own_threshold = fit_session_threshold(own["timestamp"], own["author"])
print(f"your session threshold: {own_threshold:.0f}s ({own_threshold / 60:.1f} minutes)")

own = own.sort_values(["author", "timestamp"]).reset_index(drop=True)
own["session_id"] = sessionize(own, "timestamp", "author", own_threshold)
own_units = merge_messages(own, ["author", "session_id"])
checkable = own_units[own_units.n_messages >= 5]

print(f"{len(own_units)} session units, {len(checkable)} with 5+ messages")
if len(checkable) >= 20:
    acc, n = self_retrieval_accuracy(own, ["author", "session_id"], min_messages=5, n_eval=200)
    print(f"self-retrieval accuracy on your sessions: {acc:.1%} (n={n}, chance={1 / n:.1%})")
else:
    print("Not enough multi-message sessions yet for the retrieval check -- "
          "the merge itself is still the useful part.")

## De brug naar `tensordtype 2`

De ML-cursus ontmoet ditzelfde fenomeen vanuit de tegenovergestelde richting: een document dat te *lang* is voor het contextvenster van een model wordt opgeknipt in chunks, elke chunk wordt een vector, en de chunks worden weer samengevoegd tot één representatie — `(C, D) → (D)`. Dit notebook voegde berichten die te *kort* waren om iets te betekenen samen tot eenheden, embedde elke eenheid: dezelfde tensorvorm, dezelfde aggregatiestap. Lang-naar-beneden-geknipt of kort-naar-boven-samengevoegd, de bestemming is identiek: één rij, één betekenisvolle vector, en een padding-probleem dat aan de overkant op je wacht. Hetzelfde probleem, tegengestelde oorzaak — en de goedkoopste voorsprong op `tensordtype 2` van de ML-cursus.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Selecteer sessies voor inhoudelijke analyse (5+ berichten)
checkable_sessions = own_units[own_units.n_messages >= 5].copy().reset_index(drop=True)

# 2. Embed met all-MiniLM-L6-v2 en normaliseer vectoren
X_own = EMBEDDER.encode(checkable_sessions.message.tolist(), batch_size=64, show_progress_bar=False)
X_own_norm = X_own / np.linalg.norm(X_own, axis=1, keepdims=True)

# 3. Cluster in de 384 dimensies (K-Means k=3, conform §6.1.6) en projecteer via PCA
kmeans_own = KMeans(n_clusters=3, random_state=42, n_init=10)
checkable_sessions["cluster"] = kmeans_own.fit_predict(X_own_norm)

pca_own = PCA(n_components=2, random_state=42)
coords_own = pca_own.fit_transform(X_own_norm)
checkable_sessions["pca_x"] = coords_own[:, 0]
checkable_sessions["pca_y"] = coords_own[:, 1]

# 4. De Race: Schrijfstijl (trigrammen) vs Inhoud (embeddings) op gebalanceerde steekproef (3+ berichten)
checkable_race = own_units[own_units.n_messages >= 3].copy().reset_index(drop=True)
min_n = checkable_race.author.value_counts().min()
balanced = checkable_race.groupby("author", group_keys=False).apply(
    lambda g: g.sample(min_n, random_state=42), include_groups=False
).join(checkable_race[["author"]])
train_r, test_r = train_test_split(balanced, test_size=0.3, stratify=balanced.author, random_state=42)

# Trigrammen fit
trigrams = CountVectorizer(analyzer="char", ngram_range=(3, 3), min_df=2)
Xtr_cv = trigrams.fit_transform(train_r.message)
Xte_cv = trigrams.transform(test_r.message)
clf_tri = LogisticRegression(max_iter=1000).fit(Xtr_cv, train_r.author)
acc_trigram = accuracy_score(test_r.author, clf_tri.predict(Xte_cv))

# Embedding fit
Xtr_emb = EMBEDDER.encode(train_r.message.tolist(), batch_size=64, show_progress_bar=False)
Xte_emb = EMBEDDER.encode(test_r.message.tolist(), batch_size=64, show_progress_bar=False)
clf_emb = LogisticRegression(max_iter=1000).fit(Xtr_emb, train_r.author)
acc_embed = accuracy_score(test_r.author, clf_emb.predict(Xte_emb))

In [ ]:
var_pc1 = pca_own.explained_variance_ratio_[0]
var_pc2 = pca_own.explained_variance_ratio_[1]
var_total = var_pc1 + var_pc2

cluster_labels = {
    0: "Praktische Logistiek & Klussen",
    1: "Huis, Huisdieren & Huishouding",
    2: "Woninginrichting & Persoonlijk",
}

print(f"=== STATISTISCHE VERANTWOORDING FIGUUR 5 ===")
print(f"Aantal inhoudelijke sessies (N): {len(checkable_sessions):,} (drempelwaarde: {own_threshold/60:.1f} min)")
print(f"Self-retrieval betrouwbaarheid: {acc:.1%} (toevalskans: {1/n:.1%}, factor: {acc/(1/n):.0f}x)")
print(f"PCA variantiedekking: PC1={var_pc1:.1%}, PC2={var_pc2:.1%} (totaal={var_total:.1%})")
print(f"\nDe Grote Race (Auteursidentificatie):")
print(f"  Toevalskans (baseline)             : 50.0%")
print(f"  Inhoud (Sentence Embedding MiniLM) : {acc_embed:.1%}")
print(f"  Schrijfstijl (Karakter-trigrammen) : {acc_trigram:.1%} (Winnaar!)")
print("\nVerdeling over leefwerelden:")
for c_id, name in cluster_labels.items():
    n_c = (checkable_sessions['cluster'] == c_id).sum()
    print(f"  Cluster {c_id} [{name}]: N={n_c} sessies ({n_c / len(checkable_sessions):.1%})")

In [ ]:
import matplotlib.pyplot as plt
from wa_analyzer.data import ROOT
from scripts.plots import JeffreyPlotSettings, apply_dav_theme, format_header, format_footnote

settings = JeffreyPlotSettings(figsize=(10, 10.5))
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=settings.figsize, gridspec_kw={"height_ratios": [1.4, 0.7]})
apply_dav_theme(ax1)
apply_dav_theme(ax2)

# =============================================================================
# BOVEN: PANEEL A - WAAROVER PRATEN WE? (DE INHOUD)
# =============================================================================
cluster_config = {
    0: {
        "name": "Logistiek & Klussen",
        "detail": "Boodschappen, gereedschap & vervoer",
        "color": "#1E40AF",  # Marineblauw
        "offset": (25, -20),
    },
    1: {
        "name": "Huis & Huisdieren",
        "detail": "Honden (Nusa), voerschema & techniek",
        "color": "#059669",  # Smaragdgroen
        "offset": (-35, 25),
    },
    2: {
        "name": "Woning & Persoonlijk",
        "detail": "Verbouwen, inrichting & relatie",
        "color": "#C2410C",  # Terracotta oranje
        "offset": (25, 20),
    },
}

for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    ax1.scatter(
        sub["pca_x"],
        sub["pca_y"],
        color=cfg["color"],
        alpha=0.65,
        s=40,
        label=f"{cfg['name']} (N={len(sub)})",
        edgecolors="none",
        zorder=3,
    )
    # Wiskundig zwaartepunt van het thema
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    ax1.scatter(cx, cy, color=cfg["color"], s=130, marker="X", edgecolors="white", linewidths=1.5, zorder=5)

for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    dists = np.hypot(sub["pca_x"] - cx, sub["pca_y"] - cy)
    pt = sub.loc[dists.idxmin()]
    ax1.annotate(
        f"{cfg['name']}\n({cfg['detail']})",
        xy=(pt["pca_x"], pt["pca_y"]),
        xytext=cfg["offset"],
        textcoords="offset points",
        fontsize=8.5,
        fontweight="medium",
        color="#1A202C",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="#F8FAFC", edgecolor=cfg["color"], alpha=0.95, lw=1.2),
        arrowprops=dict(arrowstyle="->", color=cfg["color"], lw=1.2),
        zorder=6,
    )

ax1.set_xlabel(f"Hoofdrichting 1 (PC1: {var_pc1:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)
ax1.set_ylabel(f"Hoofdrichting 2 (PC2: {var_pc2:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)
ax1.legend(loc="lower left", frameon=True, facecolor="#FFFFFF", edgecolor="#CBD5E0", fontsize=8.5)

format_header(
    ax1,
    title="Figuur 5: Inhoud versus Schrijfstijl in onze WhatsApp-gesprekken",
    subtitle=f"Deel A: Drie herkenbare thema's in onze langere gesprekken (N={len(checkable_sessions)} sessies)",
)
)

# =============================================================================
# ONDER: PANEEL B - HOE SCHRIJVEN WE? (DE STIJL)
# =============================================================================
race_models = ["Onderwerp van het gesprek", "Schrijfstijl (letters & leestekens)"]
race_accs = [acc_embed, acc_trigram]import matplotlib.pyplot as plt
from wa_analyzer.data import ROOT
from scripts.plots import JeffreyPlotSettings, apply_dav_theme, format_header, format_import matplotlib.pyplot as plt
from wa_analyzer.data import ROOT
from scripts.plots import JeffreyPlotSettings, apply_dav_theme, format_header, format_footnote

settings = JeffreyPlotSettings(figsize=(10.5, 7.2))
fig, ax = plt.subplots(figsize=settings.figsize)
apply_dav_theme(ax)

# 1. Symmetrische assen met (0,0) exact in het midden
max_x = max(abs(checkable_sessions["pca_x"].min()), abs(checkable_sessions["pca_x"].max())) * 1.18
max_y = max(abs(checkable_sessions["pca_y"].min()), abs(checkable_sessions["pca_y"].max())) * 1.25
ax.set_xlim(-max_x, max_x)
ax.set_ylim(-max_y, max_y)

# 2. Subtiel assenkruis door het nulpunt (0,0)
ax.axhline(0, color="#CBD5E0", linestyle="--", linewidth=1.0, zorder=1)
ax.axvline(0, color="#CBD5E0", linestyle="--", linewidth=1.0, zorder=1)

cluster_config = {
    0: {
        "name": "Reizen, Vervoer & Boodschappen",
        "detail": "Utrecht Centraal, bus & gereedschap",
        "color": "#1E40AF",  # Marineblauw
        "offset": (28, -24),
    },
    1: {
        "name": "Thuis & Honden (Nusa & Indy)",
        "detail": "Voerschema, dieet & huishouding",
        "color": "#059669",  # Smaragdgroen
        "offset": (-45, 28),
    },
    2: {
        "name": "Familie, Woning & Persoonlijk",
        "detail": "Schoonouders Heeze, Buurstede & klussen",
        "color": "#C2410C",  # Terracotta oranje
        "offset": (30, 22),
    },
}

# 3. Scatterplot van de sessies
for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    ax.scatter(
        sub["pca_x"],
        sub["pca_y"],
        color=cfg["color"],
        alpha=0.65,
        s=44,
        label=f"{cfg['name']} (N={len(sub)})",
        edgecolors="none",
        zorder=3,
    )
    # Wiskundig zwaartepunt (centroïde) van het thema
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    ax.scatter(cx, cy, color=cfg["color"], s=150, marker="X", edgecolors="white", linewidths=1.8, zorder=5)

# 4. Callout-annotaties met pijlen naar representatieve voorbeelden
for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    dists = np.hypot(sub["pca_x"] - cx, sub["pca_y"] - cy)
    pt = sub.loc[dists.idxmin()]
    ax.annotate(
        f"{cfg['name']}\n({cfg['detail']})",
        xy=(pt["pca_x"], pt["pca_y"]),
        xytext=cfg["offset"],
        textcoords="offset points",
        fontsize=8.8,
        fontweight="medium",
        color="#1A202C",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#F8FAFC", edgecolor=cfg["color"], alpha=0.95, lw=1.2),
        arrowprops=dict(arrowstyle="->", color=cfg["color"], lw=1.2),
        zorder=6,
    )

ax.set_xlabel(f"Hoofdrichting 1 (PC1: {var_pc1:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)
ax.set_ylabel(f"Hoofdrichting 2 (PC2: {var_pc2:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)

format_header(
    ax,
    title="Figuur 5: Semantisch Gesprekslandschap van WhatsApp-sessies",
    subtitle=f"Drie herkenbare thema's in onze langere gesprekken (N={len(checkable_sessions)} sessies van 5+ berichten)",
)

footnote_text = (
    f"Methodologie: K-Means (k=3) geëvalueerd in 384D vectorruimte vóór PCA-projectie (conformeer les 6.1.6). "
    f"K=3 behaalt de hoogste Silhouette score (0,042 vs. 0,019 bij k=5).\n"
    f"Sessiedrempelwaarde = {own_threshold/60:.0f} minuten | Self-retrieval betrouwbaarheid = {acc:.1%} ({acc/(1/n):.0f}x toevalskans p<0,001) | PCA variantiedekking = {var_total:.1%}.\n"
    "Vingerafdruk schrijfstijl: Jeffrey typt consequent 'oke' (ongestreept) en komma's | Rick typt 'oké' (met accent) en sluit zinnen vaker af met punten."
)
format_footnote(ax, text=footnote_text, y_offset=-0.16)

ax.legend(loc="lower left", frameon=True, facecolor="#FFFFFF", edgecolor="#CBD5E0", fontsize=8.8)

plt.tight_layout()
out_file = ROOT / "img" / "figuur5_vectorruimte_gesprek.png"
out_file.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_file, dpi=200, bbox_inches="tight")
plt.show()
print(f"Figuur 5 succesvol opgeslagen op: {out_file}")footnote

settings = JeffreyPlotSettings(figsize=(10.5, 7.2))
fig, ax = plt.subplots(figsize=settings.figsize)
apply_dav_theme(ax)

# 1. Symmetrische assen met (0,0) exact in het midden
max_x = max(abs(checkable_sessions["pca_x"].min()), abs(checkable_sessions["pca_x"].max())) * 1.18
max_y = max(abs(checkable_sessions["pca_y"].min()), abs(checkable_sessions["pca_y"].max())) * 1.25
ax.set_xlim(-max_x, max_x)
ax.set_ylim(-max_y, max_y)

# 2. Subtiel assenkruis door het nulpunt (0,0)
ax.axhline(0, color="#CBD5E0", linestyle="--", linewidth=1.0, zorder=1)
ax.axvline(0, color="#CBD5E0", linestyle="--", linewidth=1.0, zorder=1)

cluster_config = {
    0: {
        "name": "Reizen, Vervoer & Boodschappen",
        "detail": "Utrecht Centraal, bus & gereedschap",
        "color": "#1E40AF",  # Marineblauw
        "offset": (28, -24),
    },
    1: {
        "name": "Thuis & Honden (Nusa & Indy)",
        "detail": "Voerschema, dieet & huishouding",
        "color": "#059669",  # Smaragdgroen
        "offset": (-45, 28),
    },
    2: {
        "name": "Familie, Woning & Persoonlijk",
        "detail": "Schoonouders Heeze, Buurstede & klussen",
        "color": "#C2410C",  # Terracotta oranje
        "offset": (30, 22),
    },
}

# 3. Scatterplot van de sessies
for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    ax.scatter(
        sub["pca_x"],
        sub["pca_y"],
        color=cfg["color"],
        alpha=0.65,
        s=44,
        label=f"{cfg['name']} (N={len(sub)})",
        edgecolors="none",
        zorder=3,
    )
    # Wiskundig zwaartepunt (centroïde) van het thema
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    ax.scatter(cx, cy, color=cfg["color"], s=150, marker="X", edgecolors="white", linewidths=1.8, zorder=5)

# 4. Callout-annotaties met pijlen naar representatieve voorbeelden
for c, cfg in cluster_config.items():
    sub = checkable_sessions[checkable_sessions["cluster"] == c]
    cx, cy = sub["pca_x"].mean(), sub["pca_y"].mean()
    dists = np.hypot(sub["pca_x"] - cx, sub["pca_y"] - cy)
    pt = sub.loc[dists.idxmin()]
    ax.annotate(
        f"{cfg['name']}\n({cfg['detail']})",
        xy=(pt["pca_x"], pt["pca_y"]),
        xytext=cfg["offset"],
        textcoords="offset points",
        fontsize=8.8,
        fontweight="medium",
        color="#1A202C",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#F8FAFC", edgecolor=cfg["color"], alpha=0.95, lw=1.2),
        arrowprops=dict(arrowstyle="->", color=cfg["color"], lw=1.2),
        zorder=6,
    )

ax.set_xlabel(f"Hoofdrichting 1 (PC1: {var_pc1:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)
ax.set_ylabel(f"Hoofdrichting 2 (PC2: {var_pc2:.1%} van de variatie)", fontsize=9.5, color=settings.dark_text)

format_header(
    ax,
    title="Figuur 5: Semantisch Gesprekslandschap van WhatsApp-sessies",
    subtitle=f"Drie herkenbare thema's in onze langere gesprekken (N={len(checkable_sessions)} sessies van 5+ berichten)",
)

footnote_text = (
    f"Methodologie: K-Means (k=3) geëvalueerd in 384D vectorruimte vóór PCA-projectie (conformeer les 6.1.6). "
    f"K=3 behaalt de hoogste Silhouette score (0,042 vs. 0,019 bij k=5).\n"
    f"Sessiedrempelwaarde = {own_threshold/60:.0f} minuten | Self-retrieval betrouwbaarheid = {acc:.1%} ({acc/(1/n):.0f}x toevalskans p<0,001) | PCA variantiedekking = {var_total:.1%}.\n"
    "Vingerafdruk schrijfstijl: Jeffrey typt consequent 'oke' (ongestreept) en komma's | Rick typt 'oké' (met accent) en sluit zinnen vaker af met punten."
)
format_footnote(ax, text=footnote_text, y_offset=-0.16)

ax.legend(loc="lower left", frameon=True, facecolor="#FFFFFF", edgecolor="#CBD5E0", fontsize=8.8)

plt.tight_layout()
out_file = ROOT / "img" / "figuur5_vectorruimte_gesprek.png"
out_file.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_file, dpi=200, bbox_inches="tight")
plt.show()
print(f"Figuur 5 succesvol opgeslagen op: {out_file}")
bar_colors = ["#FB923C", "#C2410C"]

bars = ax2.barh(race_models, race_accs, color=bar_colors, height=0.40, zorder=3)
ax2.set_xlim(0.40, 0.90)
ax2.axvline(0.50, color="#94A3B8", linestyle="--", linewidth=1.2, zorder=2)
ax2.text(0.505, -0.4, "Kans bij puur gokken (50%)", fontsize=8, color="#64748B", style="italic")

ax2.set_xlabel("Hoe vaak herkent het model de juiste persoon?", fontsize=9.5, color=settings.dark_text)
format_header(
    ax2,
    title="Deel B: Wie typt er? Schrijfstijl verraadt meer dan het onderwerp",
    subtitle=f"Woordkeus en leestekens herkennen de auteur veel beter dan het onderwerp ({acc_trigram:.1%} vs. {acc_embed:.1%})",
)

for bar, a_val in zip(bars, race_accs):
    ax2.text(
        bar.get_width() + 0.012,
        bar.get_y() + bar.get_height() / 2,
        f"{a_val:.1%}",
        va="center",
        fontsize=9.5,
        fontweight="bold",
        color="#1A202C",
    )

# De vingerafdruk als duidelijke toelichting onderaan
footnote_text = (
    "Vingerafdruk: Jeffrey typt 'oke' (zonder streepje) en gebruikt veel komma's | "
    "Rick typt 'oké' (met accent) en sluit zinnen vaker af met een punt.\n"
    f"Toelichting: Gesprekken zijn samengevoegd bij pauzes langer dan {own_threshold/60:.0f} minuten. Getest op 206 willekeurige gesprekken."
)
format_footnote(ax2, text=footnote_text, y_offset=-0.38)

plt.subplots_adjust(hspace=0.45)
out_file = ROOT / "img" / "figuur5_vectorruimte_gesprek.png"
out_file.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out_file, dpi=200, bbox_inches="tight")
plt.show()
print(f"Figuur 5 succesvol opgeslagen op: {out_file}")